<center><h1>Blackburn_Brianna_HW3</h1></center>
<br>
<br>

Name: Brianna Blackburn
<br>
Github Username: BlackburnBC
<br>
USC ID: 9006039317

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from zipfile import ZipFile

Get the AReM Data Set

In [2]:
#Set localized file path of zipped folder
data_path = Path("C:/Users/bb76060/OneDrive - University of Southern California/"
    "Documents/Grad School/DSCI-552/HW3/"
    "activity+recognition+system+based+on+multisensor+data+fusion+arem.zip")

extract_path = data_path.parent / "AReM"

with ZipFile(data_path, "r") as zip_file:
    zip_file.extractall(extract_path)

#Establish activity categories
activity_type = ["bending1","bending2","cycling","lying","sitting","standing","walking"]

data_folders = {activity: next((folder
            for folder in extract_path.rglob(activity)
            if folder.is_dir()),None)
    for activity in activity_type}

all_files = list(extract_path.rglob("dataset*.csv"))

print("Total files:", len(all_files))

#Establish the time series data activities and column headers
sensor_series = ["avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
col_names = ["time"] + sensor_series

#Clean datafiles: remove blank and duplicate rows, uniform CSV comma separated, uniform numeric values
def data_cleaning(file):
  
    data = pd.read_csv(file, comment="#", header=None, sep=r"[,\s]+", engine="python", usecols=range(7))
    data = data.dropna(how="all")
    data.columns = col_names
    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.sort_values("time")
    data = data.drop_duplicates(subset="time")
    data = data.reset_index() #Move "time" into column positioning

    #Missing time values in Sitting folde for dataset8
    #Use interpolate function to estimate row values for time sequence 13500
    expected_time = np.arange(0, 120000, 250)
    data = data.set_index("time").reindex(expected_time)
    data[sensor_series] = data[sensor_series].interpolate(method="linear")

    return data

Total files: 88


### (b) Test and Train Data

In [3]:
#Folder paths within zipped folder
activity_folders = [folder
    for folder in extract_path.rglob("*")
    if folder.is_dir() and folder.name in activity_type]

print("Folders:", len(activity_folders))

for folder in activity_folders:
    print(folder)

bending_folders = ["bending1", "bending2"]

#Loop for reading in and saving datafiles and categorizing them between the training and test datasets
train_files = []
test_files = []

for folder in activity_folders:

    for file in folder.glob("dataset*.csv"):

        dataset_number = int(file.stem.replace("dataset", ""))

        is_bending_test = (folder.name in ["bending1", "bending2"]
            and dataset_number in [1, 2])
        is_other_test = (folder.name not in ["bending1", "bending2"]
            and dataset_number in [1, 2, 3])

        if is_bending_test or is_other_test:
            test_files.append(file)
        else:
            train_files.append(file)

print("Files:")
print("Training =", len(train_files))
print("Testing =", len(test_files))

#Sort activity type in logical numeric order for dataset structure
activity_order = {activity: position
    for position, activity in enumerate(activity_type)}

all_data = sorted(train_files + test_files,
    key=lambda file: (activity_order[file.parent.name],
        int(file.stem.replace("dataset", ""))))

#Confirm datafiles are split correctly between test and train data
print("\nTest files per activity:")
print(pd.Series([file.parent.name for file in test_files]).value_counts())

Folders: 7
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\bending1
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\bending2
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\cycling
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\lying
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\sitting
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\standing
C:\Users\bb76060\OneDrive - University of Southern California\Documents\Grad School\DSCI-552\HW3\AReM\walking
Files:
Training = 69
Testing = 19

Test files per activity:
cycling     3
lying       3
sitting     3
standing    3
walking     3
bending1    2
bending2    2
Name: count, dtype: int64


### (c) Feature Extraction

#### i. Research

Answer: Some usual time-domain features can be minimum and maximum to show then calculate range as an additional feature, median, mean/average, quartiles and interquartile ranges, standard deviation, and variance. Additional features can be trend and slope using a moving average, and also skewness and kurtosis that shows shape of range distribution. 

#### ii. Extraction

In [4]:

#Create loop to read each file, assign column headers, and create 7 statistical features
# Read each file and calculate the 42 statistical features
feature_calcs = []

test_file_set = set(test_files)

for instance, file in enumerate(all_data, start=1):
    data = data_cleaning(file)

    #Create one row for the current instance
    row = {"Instance": instance, "Activity": file.parent.name, "Dataset": file.name,
        "Split": "test" if file in test_file_set else "train"}

    #Calculate new features and append each row to bottom 
    for number, column in enumerate(sensor_series, start=1):

        row[f"min{number}"] = data[column].min()
        row[f"max{number}"] = data[column].max()
        row[f"mean{number}"] = data[column].mean()
        row[f"median{number}"] = data[column].median()
        row[f"std{number}"] = data[column].std()
        row[f"1st_quart{number}"] = data[column].quantile(0.25)
        row[f"3rd_quart{number}"] = data[column].quantile(0.75)

    feature_calcs.append(row)

#Convert all dataset and new features into dataframe
df_feature_calcs = pd.DataFrame(feature_calcs)

feature_columns = [column
    for column in df_feature_calcs.columns
    if column not in ["Instance", "Activity Type", "Dataset #", "Train/Test"]]

df_feature_calcs


,Instance,Activity,Dataset,Split,min1,max1,mean1,median1,std1,1st_quart1,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,bending1,dataset1.csv,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,bending1,dataset2.csv,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,bending1,dataset3.csv,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,bending1,dataset4.csv,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,bending1,dataset5.csv,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,walking,dataset11.csv,train,19.50,45.33,33.586875,34.25,4.650935,30.25,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,walking,dataset12.csv,train,19.75,45.50,34.322750,35.25,4.752477,31.00,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,walking,dataset13.csv,train,19.50,46.00,34.546229,35.25,4.842294,31.25,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,walking,dataset14.csv,train,23.50,46.25,34.873229,35.25,4.531720,31.75,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [5]:
#Create base sample standard deviation of features
#Created new dataframe bc i could not locate where the non-feature columns were getting in and causing error
#42 extracted numeric features df
features_data = df_feature_calcs.drop(columns=["Instance", "Activity", "Dataset", "Split"])
feature_cols = features_data.columns.tolist() #Overwrite earlier feature columns
feature_vals = features_data.to_numpy(dtype=float)

#Calculate standard deviation of each 88 instances for all 42 features
feature_std = features_data.std(ddof=1)
feature_std
print(" ")

#Selected bootstrap parameters
np.random.seed(42) 
n_bootstrap = 1000

#Loop through all feature values
confidence_results = []

for feature in features_data.columns:

    values = features_data[feature].values
    original_std = np.std(values, ddof=1)
    bootstrap_std = []

    for i in range(n_bootstrap):

        #Randomized sampling
        bootstrap_sample = np.random.choice(values, size=len(values),replace=True)

        #Each samples standard deviation
        sample_std = np.std(bootstrap_sample, ddof=1)
        bootstrap_std.append(sample_std)

    # Calculate the middle 90% of bootstrap results
    lower_bound = np.percentile(bootstrap_std, 5)
    upper_bound = np.percentile(bootstrap_std, 95)

    confidence_results.append({"Feature": feature, "Std": original_std,
        "90% Lower": lower_bound, "90% Upper": upper_bound})

bootstrap_results = pd.DataFrame(confidence_results)
print("Confidence Intervals")
bootstrap_results.round(3)

 
Confidence Intervals


,Feature,Std,90% Lower,90% Upper
0,min1,9.570,8.267,10.693
1,max1,4.394,3.370,5.293
2,mean1,5.336,4.720,5.902
3,median1,5.440,4.763,6.018
4,std1,1.772,1.571,1.953
5,1st_quart1,6.154,5.546,6.660
6,3rd_quart1,5.139,4.388,5.841
7,min2,0.000,0.000,0.000
8,max2,5.063,4.627,5.404
9,mean2,1.574,1.402,1.714


#### iv. Select Features

Answer: The first feature I would choose is standard deviation because it provides more clues into how much sensors fluctuate during an activity and thats pretty important when determining if we can confidently apply sample results. Another would be Max to show top values where performance may peak for an activity because this varies between activities. I would also choose mean as a usual significant feature because it shows what an average anticipated sensor would be and that is helpful to measure against. 

## 2. ISLR 3.7.4

### (a) Linear Train

Answer: The residual sum of squares in a simplified sense measures the difference or distance between the models predicted values and the actual observation values from the classification plane. If we know the relationship is linear, I expect the linear RSS and cubic regression RSS on this dataset to be pretty similar since the data does not contain a lot of variability/or noise. The cubic regression will not vary as much from the linear fit when fitting more flexibly. There could be small difference between model performance between linear and regression, if the dataset was less linear, then the cubic regression fit RSS would be less because it would out perform the linear model with. 

### (b) Linear Test

Answer: The linear relationship test data could impact the performance of cubic regression if it had overfitg  the training data and there exists varying noise between the training dataset and test dataset. If this is the case we would expect the linear RSS to be less than the cubic RSS.

### (c) Not Linear Train

Answer: Even though we don't know the extent of the non-linearity, the cubic regression RSS would most likely be less than the linear because the cubic model still offers more flexibility in fitting the decision boundary. So it will fit more to the actual shape of the relationship than a linear plane. 

### (d) Not Linear Testing

Answer: Even after fitting the training data, there is still uncertainty that the cubic regression would out perform the linear model. The cubic model has more risk of overfitting to the training data and there is the possibility that the random noise of the test data is different. Additionally, the shape of the curve would impact whether linear or cubic outperforms the other. The more pronounced the curve the less cubic RSS would be, but if the nonlinear relationship is not so much a cubic curve the linear could fit better theoretically. So with simple conditions around this scenario, we don't know.